# Built-in and custom disk flux profiles

This notebook compares the default Novikov--Thorne profile, the built-in Shakura--Sunyaev profile, and a custom profile with additional physical parameters.

In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import torch

import microcaustics as mc

torch.set_default_dtype(torch.float64)

A custom callable always receives `(radius_rg, spin, isco_rg)`. Any additional parameters are stored on the callable itself. The optional `name` and `metadata()` members make those choices visible in source provenance.

In [ ]:
@dataclass(frozen=True)
class WindSuppressedFlux:
    wind_index: float
    transition_rg: float
    name: str = "wind-suppressed"

    def __call__(self, radius_rg, spin, isco_rg):
        base = mc.shakura_sunyaev_flux_factor(radius_rg, spin, isco_rg)
        suppression = (1 + radius_rg / self.transition_rg) ** (-self.wind_index)
        return base * suppression

    def metadata(self):
        return {
            "wind_index": self.wind_index,
            "transition_rg": self.transition_rg,
        }

custom_profile = WindSuppressedFlux(wind_index=0.35, transition_rg=50.0)

In [ ]:
spin = torch.tensor(0.7)
isco_rg = mc.kerr_isco_radius(spin)
radius_rg = torch.logspace(torch.log10(isco_rg * 1.001), 3.0, 1000)

profiles = {
    "Novikov--Thorne": mc.novikov_thorne_flux_factor(radius_rg, spin, isco_rg),
    "Shakura--Sunyaev": mc.shakura_sunyaev_flux_factor(radius_rg, spin, isco_rg),
    "Wind-suppressed": custom_profile(radius_rg, spin, isco_rg),
}

fig, ax = plt.subplots(figsize=(6.5, 4.0), constrained_layout=True)
for label, profile in profiles.items():
    ax.loglog(radius_rg, profile / profile.max(), label=label)
ax.set(xlabel=r"Radius [$r_g$]", ylabel="Normalized viscous flux factor")
ax.legend(frameon=False)
plt.show()

Built-in profiles use their matching analytic radiative efficiency when `radiative_efficiency=None`. A custom profile must choose an explicit scalar, built-in efficiency name, or efficiency callable.

In [ ]:
common = dict(
    black_hole_mass_solar=1.0e9,
    eddington_ratio=0.1,
    spin=spin,
)
temperature4 = {
    "Novikov--Thorne": mc.thin_disk_temperature4(
        radius_rg, viscous_flux_profile="novikov-thorne", **common
    )[0],
    "Shakura--Sunyaev": mc.thin_disk_temperature4(
        radius_rg, viscous_flux_profile="shakura-sunyaev", **common
    )[0],
    "Wind-suppressed": mc.thin_disk_temperature4(
        radius_rg,
        viscous_flux_profile=custom_profile,
        radiative_efficiency=0.1,
        **common,
    )[0],
}

fig, ax = plt.subplots(figsize=(6.5, 4.0), constrained_layout=True)
for label, value in temperature4.items():
    ax.loglog(radius_rg, value.pow(0.25), label=label)
ax.set(xlabel=r"Radius [$r_g$]", ylabel=r"$T_\mathrm{eff}$ [K]")
ax.legend(frameon=False)
plt.show()

In [ ]:
model = mc.ThinDiskModel(
    black_hole_mass_solar=1.0e9,
    eddington_ratio=0.1,
    bands_angstrom={"g": 4827.0, "i": 7546.0},
    source_redshift=1.5,
    viscous_flux_profile=custom_profile,
    radiative_efficiency=0.1,
    source_grid_shape=128,
)
source = model.pixelate(H0=70.0, Om0=0.3)
source.metadata()["viscous_flux_profile_metadata"]